# 質問応答を完全一致と F1 で評価する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/20_qa/qa_em_f1.ipynb)

NLP クックブック 20-03 ／ 質問応答

質問応答の出力を、正解と完全に一致したか (EM)、部分的にどれくらい重なっているか (F1) で評価したい。

In [1]:
!pip install -q fugashi unidic-lite

In [2]:
import re
import unicodedata
from collections import Counter
from fugashi import Tagger

tagger = Tagger("-Owakati")

def normalize(text):
    text = unicodedata.normalize("NFKC", text).lower()
    return re.sub(r"[\s。、．，,.!?！？「」『』()（）]", "", text)

def exact_match(pred, golds):
    return float(any(normalize(pred) == normalize(g) for g in golds))

def token_f1(pred, golds, unit="char"):
    """正解が複数あるときは、最も高い F1 を採用する"""
    tok = (lambda s: list(normalize(s))) if unit == "char" else (lambda s: tagger.parse(s).split())
    best = 0.0
    for g in golds:
        p, t = tok(pred), tok(g)
        common = sum((Counter(p) & Counter(t)).values())
        if common:
            prec, rec = common / len(p), common / len(t)
            best = max(best, 2 * prec * rec / (prec + rec))
    return best

examples = [("富士山", ["富士山"]), ("富士山です。", ["富士山"]), ("3776 メートル", ["3776m", "3776メートル"]),
            ("１９４５年", ["1945年", "1945"]), ("夏目漱石と森鴎外", ["夏目漱石"]), ("わかりません", ["東京"])]
for pred, golds in examples:
    print(f"EM {exact_match(pred, golds):.0f}  F1(文字) {token_f1(pred, golds):.2f}  F1(単語) {token_f1(pred, golds, 'word'):.2f}  {pred!r} / {golds}")

EM 1  F1(文字) 1.00  F1(単語) 1.00  '富士山' / ['富士山']
EM 0  F1(文字) 0.75  F1(単語) 0.67  '富士山です。' / ['富士山']
EM 1  F1(文字) 1.00  F1(単語) 1.00  '3776 メートル' / ['3776m', '3776メートル']
EM 1  F1(文字) 1.00  F1(単語) 0.50  '１９４５年' / ['1945年', '1945']
EM 0  F1(文字) 0.67  F1(単語) 0.57  '夏目漱石と森鴎外' / ['夏目漱石']
EM 0  F1(文字) 0.00  F1(単語) 0.00  'わかりません' / ['東京']


- 英語の SQuAD では、冠詞と句読点を除いて小文字化し、空白区切りの単語で F1 を計算する。日本語（JSQuAD など）では文字単位の F1 が使われる。
- LLM の出力は「富士山です。」のように文になりがちで、EM が不当に低くなる。プロンプトで形式を指定するか、「正解を含むか」での判定や LLM による判定を併用する。